In [15]:
#!/usr/bin/env python3
"""Concise GPS and IMU Data Plotter using Plotly - Saves to HTML"""

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import csv
from datetime import datetime
import os

def load_csv(file, converters):
    """Load CSV with type conversions"""
    data = {k: [] for k in converters.keys()}
    with open(file, 'r') as f:
        for row in csv.DictReader(f):
            for key, converter in converters.items():
                data[key].append(converter(row[key]))
    print(f"Loaded {len(data[list(data.keys())[0]])} records from {file}")
    return data

def load_gps(file='gps_data.csv'):
    """Load GPS data"""
    return load_csv(file, {
        'timestamp': float, 'latitude': float, 'longitude': float, 
        'altitude': float, 'seq': int, 'frame_id': str, 
        'position_covariance_type': int, 'status': int
    })

def load_imu(file='imu_data.csv'):
    """Load IMU data"""
    return load_csv(file, {
        'timestamp': float, 'seq': int, 'frame_id': str,
        'orientation_x': float, 'orientation_y': float, 
        'orientation_z': float, 'orientation_w': float,
        'angular_velocity_x': float, 'angular_velocity_y': float, 
        'angular_velocity_z': float,
        'linear_acceleration_x': float, 'linear_acceleration_y': float, 
        'linear_acceleration_z': float
    })

print("Loading data...")
gps = load_gps('gps_data.csv')
imu = load_imu('imu_data.csv')


Loading data...
Loaded 30758 records from gps_data.csv
Loaded 616322 records from imu_data.csv


In [16]:
#!/usr/bin/env python3
"""Estimate and remove IMU biases - Fixed"""
import numpy as np

def estimate_imu_bias(imu, duration=5.0):
    """Estimate IMU biases from stationary period"""
    idx = [i for i, t in enumerate(imu['timestamp']) if t - imu['timestamp'][0] <= duration]
    
    biases = {
        'acc': np.array([np.mean([imu[f'linear_acceleration_{ax}'][i] for i in idx]) for ax in ['x','y','z']]),
        'gyro': np.array([np.mean([imu[f'angular_velocity_{ax}'][i] for i in idx]) for ax in ['x','y','z']])
    }
    
    print(f"Acc bias: {biases['acc']}, Gyro bias: {biases['gyro']}")
    return biases

def remove_imu_bias(imu, biases):
    """Remove biases from IMU"""
    imu_clean = imu.copy()
    for i, ax in enumerate(['x','y','z']):
        imu_clean[f'linear_acceleration_{ax}'] = [a - biases['acc'][i] for a in imu[f'linear_acceleration_{ax}']]
        imu_clean[f'angular_velocity_{ax}'] = [w - biases['gyro'][i] for w in imu[f'angular_velocity_{ax}']]
    return imu_clean

# Re-run:
biases = estimate_imu_bias(imu)
imu_unbiased = remove_imu_bias(imu, biases)


Acc bias: [-5.17757394e-01  4.86788065e-03 -9.53152827e+00], Gyro bias: [-0.00157002 -0.00739549 -0.03625914]


In [17]:
#!/usr/bin/env python3
"""GPS to ENU conversion with status - FIXED"""
import numpy as np

# Check raw GPS data
print("Raw GPS data (first 3 points):")
print(f"Latitude: {gps['latitude'][:3]}")
print(f"Longitude: {gps['longitude'][:3]}")
print(f"Altitude: {gps['altitude'][:3]}")

def gps_to_enu_with_status(gps):
    """Convert GPS lat/lon to ENU and include status"""
    # Find valid (non-NaN) GPS points
    valid_indices = [i for i, lat in enumerate(gps['latitude']) if not np.isnan(lat)]
    
    # Get valid GPS data
    lats = np.array([gps['latitude'][i] for i in valid_indices])
    lons = np.array([gps['longitude'][i] for i in valid_indices])
    alts = np.array([gps['altitude'][i] for i in valid_indices])
    timestamps = [gps['timestamp'][i] for i in valid_indices]
    statuses = [gps['status'][i] for i in valid_indices]
    
    # Reference point (first valid GPS)
    lat0, lon0, alt0 = lats[0], lons[0], alts[0]
    R = 6371000.0  # Earth radius (m)
    
    # Convert to ENU
    x = np.radians(lons - lon0) * R * np.cos(np.radians(lat0))
    y = np.radians(lats - lat0) * R
    z = alts - alt0
    
    return {
        'timestamp': timestamps,
        'x': x.tolist(),
        'y': y.tolist(),
        'z': z.tolist(),
        'status': statuses
    }

# Convert GPS to ENU with status
gps_enu = gps_to_enu_with_status(gps)

print(f"\n=== GPS ENU Conversion Complete ===")
print(f"Total valid points: {len(gps_enu['timestamp'])}")
print(f"First 3 positions: x={gps_enu['x'][:3]}, y={gps_enu['y'][:3]}")
print(f"First 10 status values: {gps_enu['status'][:10]}")
print(f"Unique status values: {set(gps_enu['status'])}")

# Verify all fields have same length
print(f"\n=== Field Lengths ===")
for key in gps_enu.keys():
    print(f"{key}: {len(gps_enu[key])}")

# Check that all lengths match
lengths = [len(gps_enu[key]) for key in gps_enu.keys()]
if len(set(lengths)) == 1:
    print("\n✓ All fields have matching lengths")
else:
    print("\n✗ WARNING: Field lengths don't match!")

Raw GPS data (first 3 points):
Latitude: [nan, nan, nan]
Longitude: [nan, nan, nan]
Altitude: [nan, nan, nan]

=== GPS ENU Conversion Complete ===
Total valid points: 21871
First 3 positions: x=[0.0, 0.0, 0.04109417557726747], y=[0.0, 0.5003771694272708, 0.9080919010081139]
First 10 status values: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Unique status values: {0, 1}

=== Field Lengths ===
timestamp: 21871
x: 21871
y: 21871
z: 21871
status: 21871

✓ All fields have matching lengths


In [18]:
#!/usr/bin/env python3
"""Save and Load GPS and IMU data to/from CSV"""
import csv
import numpy as np

# ============================================================================
# SAVE FUNCTIONS
# ============================================================================

def save_gps_enu_to_csv(gps_enu, filename='gps_enu.csv'):
    """Save gps_enu dict to CSV file"""
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        
        # Write header
        writer.writerow(['timestamp', 'x', 'y', 'z', 'status'])
        
        # Write data
        for i in range(len(gps_enu['timestamp'])):
            writer.writerow([
                gps_enu['timestamp'][i],
                gps_enu['x'][i],
                gps_enu['y'][i],
                gps_enu['z'][i],
                gps_enu['status'][i]
            ])
    
    print(f"Saved {len(gps_enu['timestamp'])} GPS ENU points to {filename}")

def save_imu_unbiased_to_csv(imu_unbiased, filename='imu_unbiased.csv'):
    """Save imu_unbiased dict to CSV file"""
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        
        # Write header
        writer.writerow([
            'timestamp', 'seq', 'frame_id',
            'orientation_x', 'orientation_y', 'orientation_z', 'orientation_w',
            'angular_velocity_x', 'angular_velocity_y', 'angular_velocity_z',
            'linear_acceleration_x', 'linear_acceleration_y', 'linear_acceleration_z'
        ])
        
        # Write data
        for i in range(len(imu_unbiased['timestamp'])):
            writer.writerow([
                imu_unbiased['timestamp'][i],
                imu_unbiased['seq'][i],
                imu_unbiased['frame_id'][i],
                imu_unbiased['orientation_x'][i],
                imu_unbiased['orientation_y'][i],
                imu_unbiased['orientation_z'][i],
                imu_unbiased['orientation_w'][i],
                imu_unbiased['angular_velocity_x'][i],
                imu_unbiased['angular_velocity_y'][i],
                imu_unbiased['angular_velocity_z'][i],
                imu_unbiased['linear_acceleration_x'][i],
                imu_unbiased['linear_acceleration_y'][i],
                imu_unbiased['linear_acceleration_z'][i]
            ])
    
    print(f"Saved {len(imu_unbiased['timestamp'])} IMU points to {filename}")


In [19]:
# Save data
save_gps_enu_to_csv(gps_enu, 'gps_enu.csv')
save_imu_unbiased_to_csv(imu_unbiased, 'imu_unbiased.csv')


Saved 21871 GPS ENU points to gps_enu.csv
Saved 616322 IMU points to imu_unbiased.csv
